In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("GoldMarketExplorer")
    .getOrCreate()
)

file_paths = {
    "regional_market": "/content/regional_market.csv",
    "gold_daily_prices": "/content/gold_daily_prices.csv",
    "gold_transactions": "/content/gold_transactions.csv",
    "market_events": "/content/market_events.csv"
}

dfs = {}

for name, path in file_paths.items():
    dfs[name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

    print(f"{name}: loaded successfully")

regional_market: loaded successfully
gold_daily_prices: loaded successfully
gold_transactions: loaded successfully
market_events: loaded successfully


**regional_markets.csv**

In [2]:
df_region = dfs["regional_market"]

In [3]:
df_region.printSchema()
df_region.show(truncate=False)


root
 |-- region_id: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- making_charge_pct: string (nullable = true)

+---------+-----------+-----------+-----------------+
|region_id|region_name|state      |making_charge_pct|
+---------+-----------+-----------+-----------------+
|RG01     |Hyderabad  |Telangana  |12               |
|RG02     |Mumbai     |Maharashtra|12               |
|RG03     |Delhi      |Delhi      |5                |
|RG04     |Chennai    |Tamil Nadu |8                |
|RG05     |Kolkata    |West Bengal|10               |
|RG06     |Bengaluru  |Karnataka  |12               |
|RG07     |Pune       |Maharashtra|-3               |
|RG08     |Jaipur     |Rajasthan  |5                |
+---------+-----------+-----------+-----------------+



In [4]:
print("Rows:", df_region.count())
print("Columns:", len(df_region.columns))

Rows: 8
Columns: 4


In [5]:
from pyspark.sql import functions as F

# 1. Check NULL values
df_region.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_region.columns
]).show()

+---------+-----------+-----+-----------------+
|region_id|region_name|state|making_charge_pct|
+---------+-----------+-----+-----------------+
|        0|          0|    0|                0|
+---------+-----------+-----+-----------------+



In [6]:
# 2. Check blank values
df_region.select([
    F.count(
        F.when(F.trim(F.col(c)) == "", c)
    ).alias(c)
    for c in df_region.columns
]).show()

+---------+-----------+-----+-----------------+
|region_id|region_name|state|making_charge_pct|
+---------+-----------+-----+-----------------+
|        0|          0|    0|                0|
+---------+-----------+-----+-----------------+



In [7]:
# 3. Check duplicate region IDs
df_region.groupBy("region_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show()

+---------+-----+
|region_id|count|
+---------+-----+
+---------+-----+



In [8]:
# 4. Check distinct region IDs
print("Total rows:", df_region.count())
print("Distinct region IDs:", df_region.select("region_id").distinct().count())

Total rows: 8
Distinct region IDs: 8


In [9]:
# handling the negative making charges
#Retain the Pune (RG07) record to preserve the region reference,
#but flag its negative making-charge percentage (-3%) as invalid.
#Set the percentage to NULL in the curated dataset, exclude it from making-charge calculations, and
#record the original value and reason in the data-quality report for further review.

**gold_daily_prices.csv**

In [10]:
df_prices = dfs["gold_daily_prices"]

df_prices.printSchema()
df_prices.show(10, truncate=False)

print("Rows:", df_prices.count())
print("Columns:", len(df_prices.columns))

root
 |-- price_date: string (nullable = true)
 |-- market: string (nullable = true)
 |-- price_per_10g_inr: string (nullable = true)
 |-- usd_inr: string (nullable = true)
 |-- global_gold_usd_oz: string (nullable = true)
 |-- source_label: string (nullable = true)

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|2025-01-01|India |2659.09          |83.08  |2899.9            |Daily Close |
|2025-01-02|India |2684.83          |85.78  |3252.48           |daily_close |
|2025-01-03|India |2659.42          |85.34  |2985.6            |daily_close |
|2025-01-04|India |2685.03          |85.21  |3031.05           |NULL        |
|2025-01-05|India |2681.74          |86.1   |3276.16           |market close|
|2025-01-06|India |2704.02          |84.25  |2841.41           |Daily Close |
|2025-01-07|India |2686.14    

In [11]:
from pyspark.sql import functions as F

# NULL values
df_prices.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_prices.columns
]).show()

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|         0|     0|                0|      0|                 0|          88|
+----------+------+-----------------+-------+------------------+------------+



In [12]:

# Empty or whitespace-only values
df_prices.select([
    F.count(
        F.when(F.trim(F.col(c)) == "", c)
    ).alias(c)
    for c in df_prices.columns
]).show()

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|         0|     0|                0|      0|                 0|           0|
+----------+------+-----------------+-------+------------------+------------+



In [13]:
#We'll preserve the NULL value rather than inventing a source label.
#We can also report the 88 missing values in our data-quality summary.

In [14]:
# Check exact duplicate records
print("Total rows:", df_prices.count())
print("Distinct rows:", df_prices.distinct().count())

# Check duplicate date-market combinations
df_prices.groupBy("price_date", "market") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

Total rows: 367
Distinct rows: 365
+----------+------+-----+
|price_date|market|count|
+----------+------+-----+
|2025-01-21|India |2    |
|2025-04-01|India |2    |
+----------+------+-----+



In [15]:
df_prices.filter(
    F.col("price_date").isin("2025-01-21", "2025-04-01")
).orderBy("price_date").show(truncate=False)

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|2025-01-21|India |2709.21          |87.8   |2823.8            |daily_close |
|2025-01-21|India |2709.21          |87.8   |2823.8            |daily_close |
|2025-04-01|India |2863.76          |86.39  |2617.05           |Daily Close |
|2025-04-01|India |2863.76          |86.39  |2617.05           |Daily Close |
+----------+------+-----------------+-------+------------------+------------+



In [16]:
# Exact duplicate records will be removed using all columns, retaining one copy of each record.
# Since the duplicate records contain identical values across all columns,
 # removing the redundant copies does not result in any information loss.

In [17]:
# Validate numeric values
numeric_cols = [
    "price_per_10g_inr",
    "usd_inr",
    "global_gold_usd_oz"
]

for c in numeric_cols:
    print(f"--- {c} ---")

    df_prices.select(
        F.min(c).alias("minimum"),
        F.max(c).alias("maximum")
    ).show()

--- price_per_10g_inr ---
+-------+-------+
|minimum|maximum|
+-------+-------+
|  -2500|unknown|
+-------+-------+

--- usd_inr ---
+-------+-------+
|minimum|maximum|
+-------+-------+
|   82.0|  87.98|
+-------+-------+

--- global_gold_usd_oz ---
+-------+-------+
|minimum|maximum|
+-------+-------+
|2553.97|3297.35|
+-------+-------+



In [18]:
for c in numeric_cols:
    print(f"--- Invalid values in {c} ---")

    df_prices.filter(
        F.col(c).isNotNull() &
        ~F.trim(F.col(c)).rlike(r"^[+-]?\d+(\.\d+)?$")
    ).select(c).distinct().show(truncate=False)

--- Invalid values in price_per_10g_inr ---
+-----------------+
|price_per_10g_inr|
+-----------------+
|unknown          |
+-----------------+

--- Invalid values in usd_inr ---
+-------+
|usd_inr|
+-------+
+-------+

--- Invalid values in global_gold_usd_oz ---
+------------------+
|global_gold_usd_oz|
+------------------+
+------------------+



In [19]:
df_prices.filter(
    F.expr("try_cast(price_per_10g_inr AS DOUBLE)") < 0
).show(truncate=False)

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|2025-02-14|India |-2500            |85.57  |2833.52           |market close|
+----------+------+-----------------+-------+------------------+------------+



In [20]:
#The record dated 2025-02-14 contains a negative gold price (-2500), which violates the business rule that gold prices must be positive.
#The record will be quarantined because its price cannot be used reliably in price-trend calculations.

In [21]:
# finding the exact unknown record
df_prices.filter(
    F.lower(F.trim(F.col("price_per_10g_inr"))) == "unknown"
).show(truncate=False)

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|2025-01-31|India |unknown          |86.46  |2615.43           |market close|
+----------+------+-----------------+-------+------------------+------------+



In [22]:

df_prices.filter(
    F.expr(
        "try_to_timestamp(price_date, 'yyyy-MM-dd') IS NULL"
    )
).show(truncate=False)

+----------+------+-----------------+-------+------------------+------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|
+----------+------+-----------------+-------+------------------+------------+
|2025/02/24|India |2761.88          |83.1   |2904.72           |daily_close |
+----------+------+-----------------+-------+------------------+------------+



In [23]:
# The date 2025/02/24 uses an inconsistent date format.
#It will be normalized to 2025-02-24 during transformation, preserving the record.

In [24]:
df_prices.select(
    F.min("price_date").alias("minimum_date"),
    F.max("price_date").alias("maximum_date")
).show()

+------------+------------+
|minimum_date|maximum_date|
+------------+------------+
|  2025-01-01|  2025/02/24|
+------------+------------+



In [25]:
# checking market label consistency
df_prices.select(
    "market"
).distinct().show(truncate=False)

# Leading and trailing whitespace will be removed from the market
# column using trim() to maintain consistent market values.

+------+
|market|
+------+
|India |
+------+



In [26]:
# source labels
df_prices.select(
    "source_label"
).distinct().show(truncate=False)

+------------+
|source_label|
+------------+
|daily_close |
|market close|
|Daily Close |
|NULL        |
+------------+



In [27]:
# Source labels will be standardized to a consistent lowercase format with normalized whitespace and separators.
# Missing source labels will remain NULL because the original source cannot be determined.

**gold_transactions.csv**

In [28]:
df_transactions = dfs["gold_transactions"]

df_transactions.printSchema()
df_transactions.show(10, truncate=False)

print("Rows:", df_transactions.count())
print("Columns:", len(df_transactions.columns))

root
 |-- transaction_id: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- purity: string (nullable = true)
 |-- weight_grams: string (nullable = true)
 |-- quoted_rate_per_10g: string (nullable = true)
 |-- making_charge_pct: string (nullable = true)
 |-- transaction_status: string (nullable = true)

+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000001      |2025-10-05|RG03     |coin        |24K   |4           |2968.21            |0.12             |completed         |
|GT000002      |2025-11-22|RG03     |necklace    |24K   |100         |3185.62 

In [29]:
# NULL values
df_transactions.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in df_transactions.columns
]).show()

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|             0|        0|        0|           0|   530|          11|                  3|               11|                 0|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+



In [30]:
# Empty or whitespace-only values
df_transactions.select([
    F.count(
        F.when(F.trim(F.col(c)) == "", c)
    ).alias(c)
    for c in df_transactions.columns
]).show()

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|             0|        0|        0|           0|     0|           0|                  0|                0|                 0|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+



In [31]:
# Records with missing non-key attributes will not automatically be quarantined.
# They will be retained where possible and excluded only from analyses that
# require the missing attributes.



In [32]:
# Missing making-charge percentages will be retained as NULL. Although the
# regional_market dataset contains regional making-charge percentages, these
# will not be used as replacements because transaction-level charges can vary
# within the same region. No confirmed business rule establishes the regional
# percentage as a default for missing transaction values.

In [33]:
# checking the uniqueness of transaction id
print("Total rows:", df_transactions.count())

print(
    "Distinct transaction IDs:",
    df_transactions.select("transaction_id").distinct().count()
)

df_transactions.groupBy("transaction_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

Total rows: 2602
Distinct transaction IDs: 2600
+--------------+-----+
|transaction_id|count|
+--------------+-----+
|GT000056      |2    |
|GT000009      |2    |
+--------------+-----+



In [34]:
# inspecting those complete records
df_transactions.filter(
    F.col("transaction_id").isin("GT000056", "GT000009")
).orderBy("transaction_id").show(truncate=False)

+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000009      |2025-11-06|RG04     |coin        |22k   |100         |3197.36            |0.1              |Completed         |
|GT000009      |2025-11-06|RG04     |coin        |22k   |100         |3197.36            |0.1              |Completed         |
|GT000056      |2025-06-24|RG04     |ring        |24K   |50          |TBD                |0.1              |returned          |
|GT000056      |2025-06-24|RG04     |ring        |24K   |50          |2966.47            |0.1              |returned          |
+--------------+----------+---------+------------+------+------------+-------------------+--------------

In [35]:
# GT000009 contains an exact duplicate across all columns.
# Retain one copy and remove the duplicate to avoid double-counting.

In [36]:
# GT000056 has conflicting records with different quoted rates ('TBD'
# and 2966.47). Since the correct value cannot be established from the
# available data, flag the records for investigation rather than
# arbitrarily selecting one.

In [37]:
# checking the malformed dates
df_transactions.filter(
    F.expr("try_to_timestamp(sale_date, 'yyyy-MM-dd')").isNull()
).show(truncate=False)

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000045      |unknown  |RG06     |ring        |NULL  |5           |3004.14            |0.03             |completed         |
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+



In [38]:
# GT000045 contains an invalid sale_date value ('unknown').
# Since the transaction date cannot be reliably determined, this record
# will be quarantined from date-dependent processing and reported with
# the reason INVALID_SALE_DATE. The original record will be preserved.

In [39]:
# checking missing & malformed weights
df_transactions.filter(
    F.col("weight_grams").isNotNull() &
    (
        ~F.trim(F.col("weight_grams")).rlike(r"^[+-]?\d+(\.\d+)?$") |
        (F.expr("try_cast(weight_grams AS DOUBLE)") <= 0)
    )
).show(truncate=False)

+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000011      |2025-06-06|RG08     |bar         |NULL  |unknown     |2898.42            |0.08             |returned          |
|GT000024      |2025-04-16|RG01     |necklace    |24K   |-10         |2653.29            |0.12             |returned          |
|GT000120      |2025-08-30|RG06     |ring        |22K   |unknown     |2634.91            |0.1              |completed         |
|GT000172      |2025-11-28|RG08     |bar         |22k   |N/A         |2604.91            |0.12             |pending           |
|GT000183      |2025-03-24|RG06     |chain       |22k   |N/A         |2558.19            |0.03          

In [40]:
# Non-numeric weight values such as 'unknown', 'N/A' and '??' will be
# converted to NULL in the curated dataset. The original values will be
# preserved in the quality report. These transactions will be excluded
# from calculations requiring gold weight.

In [41]:
# Negative gold weights are considered invalid because transaction
# weight cannot be negative. Such records will be flagged for investigation
# and excluded from weight-dependent calculations.

In [42]:
from pyspark.sql import functions as F

df_transactions.filter(
    F.col("weight_grams").isNotNull() &
    ~F.trim(F.col("weight_grams")).rlike(r"^[+-]?\d+(\.\d+)?$")
).groupBy("weight_grams").count().show(truncate=False)

df_transactions.filter(
    F.expr("try_cast(weight_grams AS DOUBLE)") < 0
).show(truncate=False)

+------------+-----+
|weight_grams|count|
+------------+-----+
|unknown     |11   |
|N/A         |16   |
|??          |10   |
+------------+-----+

+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000024      |2025-04-16|RG01     |necklace    |24K   |-10         |2653.29            |0.12             |returned          |
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+



In [43]:
# GT000024 contains a negative weight of -10 grams.
# Since gold weight cannot be negative, this record will be flagged
# as invalid and quarantined from valid transaction processing.

In [44]:
# investigating quoted_rate_per_10g
df_transactions.filter(
    F.col("quoted_rate_per_10g").isNotNull() &
    (
        ~F.trim(F.col("quoted_rate_per_10g")).rlike(r"^[+-]?\d+(\.\d+)?$") |
        (F.expr("try_cast(quoted_rate_per_10g AS DOUBLE)") <= 0)
    )
).show(50, truncate=False)

+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+
|GT000036      |2025-10-17|RG08     |necklace    |22K   |20          |??                 |0.1              |completed         |
|GT000056      |2025-06-24|RG04     |ring        |24K   |50          |TBD                |0.1              |returned          |
|GT000066      |2025-02-20|RG01     |bar         |22k   |5           |N/A                |0.1              |Completed         |
|GT000080      |2025-04-21|RG03     |chain       |18K   |8           |unknown            |0.05             |Completed         |
|GT000117      |2025-09-04|RG02     |bar         |NULL  |100         |unknown            |0.1           

In [45]:
# Quoted-rate validation identified non-numeric placeholders such as
# 'unknown', 'N/A', '??' and 'TBD'.
# These values will be converted to NULL in the curated dataset,
# while preserving the original values in the quality report.
# Daily gold prices will not be used as replacements because the
# transaction-level quoted rate may differ from the daily market price.
# Such transactions will be excluded from rate-dependent calculations.

In [46]:
df_transactions.filter(
    F.expr("try_cast(quoted_rate_per_10g AS DOUBLE)") <= 0
).show(truncate=False)

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+



In [47]:
# validate making_charges_pct
df_transactions.filter(
    F.col("making_charge_pct").isNotNull() &
    ~F.trim(F.col("making_charge_pct")).rlike(r"^[+-]?\d+(\.\d+)?$")
).groupBy("making_charge_pct").count().show(truncate=False)

+-----------------+-----+
|making_charge_pct|count|
+-----------------+-----+
|unknown          |6    |
|N/A              |5    |
|??               |5    |
+-----------------+-----+



In [48]:
# Making-charge validation identified 16 non-numeric placeholders:
# 'unknown' (6), 'N/A' (5), and '??' (5).
# These values will be converted to NULL in the curated dataset,
# while preserving the original values in the quality report.
# Regional making-charge percentages will not be used as replacements
# because transaction-level charges may vary within the same region.

In [50]:
# checking numeric making charges values which are 0 or negative
df_transactions.filter(
    F.expr("try_cast(making_charge_pct AS DOUBLE)") < 0
).show(truncate=False)

df_transactions.filter(
    F.expr("try_cast(making_charge_pct AS DOUBLE)") == 0
).show(truncate=False)

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+---

In [51]:
# validating purity
df_transactions.groupBy("purity") \
    .count() \
    .orderBy("purity") \
    .show(truncate=False)

+------+-----+
|purity|count|
+------+-----+
|NULL  |530  |
|18K   |567  |
|22K   |497  |
|22k   |487  |
|24K   |521  |
+------+-----+



In [52]:
# Purity validation identified inconsistent capitalization:
# '22K' and '22k' represent the same purity category.
# Standardize purity labels to uppercase to maintain consistency.
# Missing purity values (530 records) will remain NULL because
# purity cannot be reliably inferred from other transaction attributes.

In [53]:
# validate product_type
df_transactions.groupBy("product_type") \
    .count() \
    .orderBy("product_type") \
    .show(truncate=False)

+------------+-----+
|product_type|count|
+------------+-----+
|bangle      |451  |
|bar         |419  |
|chain       |426  |
|coin        |441  |
|necklace    |460  |
|ring        |405  |
+------------+-----+



In [54]:
# validate transaction_status
df_transactions.groupBy("transaction_status") \
    .count() \
    .orderBy("transaction_status") \
    .show(truncate=False)

+------------------+-----+
|transaction_status|count|
+------------------+-----+
|Completed         |420  |
|cancelled         |442  |
|completed         |856  |
|pending           |462  |
|returned          |422  |
+------------------+-----+



In [55]:
# Transaction-status profiling identified inconsistent capitalization:
# 'Completed' and 'completed' represent the same status.
# Standardize all status values to lowercase to maintain consistency.
# The resulting categories will be completed, cancelled, pending and returned.

In [56]:
# validate region_id
df_transactions.groupBy("region_id") \
    .count() \
    .orderBy("region_id") \
    .show(truncate=False)

+---------+-----+
|region_id|count|
+---------+-----+
|RG01     |350  |
|RG02     |339  |
|RG03     |302  |
|RG04     |319  |
|RG05     |321  |
|RG06     |331  |
|RG07     |324  |
|RG08     |315  |
|RG99     |1    |
+---------+-----+



In [57]:
# Region IDs are present in the transaction dataset.
# Referential integrity against regional_market will be validated
# separately during cross-file validation.

In [58]:
string_columns = [
    "transaction_id",
    "sale_date",
    "region_id",
    "product_type",
    "purity",
    "weight_grams",
    "quoted_rate_per_10g",
    "making_charge_pct",
    "transaction_status"
]

for column in string_columns:
    count = df_transactions.filter(
        F.col(column).isNotNull() &
        (F.col(column) != F.trim(F.col(column)))
    ).count()

    print(f"{column}: {count} records with extra whitespace")

transaction_id: 0 records with extra whitespace
sale_date: 0 records with extra whitespace
region_id: 0 records with extra whitespace
product_type: 0 records with extra whitespace
purity: 0 records with extra whitespace
weight_grams: 0 records with extra whitespace
quoted_rate_per_10g: 0 records with extra whitespace
making_charge_pct: 0 records with extra whitespace
transaction_status: 0 records with extra whitespace


In [59]:
df_transactions.filter(
    ~F.col("transaction_id").rlike(r"^GT\d{6}$")
).show(truncate=False)

+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
|transaction_id|sale_date|region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+
+--------------+---------+---------+------------+------+------------+-------------------+-----------------+------------------+



**market_events.csv**

In [60]:
df_events = dfs["market_events"]

df_events.printSchema()
df_events.show(10, truncate=False)

print("Total rows:", df_events.count())
print("Total columns:", len(df_events.columns))

root
 |-- event_id: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- notes: string (nullable = true)

+--------+----------+------------------+---------+----------------+
|event_id|event_date|event_type        |region_id|notes           |
+--------+----------+------------------+---------+----------------+
|ME00001 |2025-04-21|wedding_season    |RG02     |local promotion |
|ME00002 |2025-02-04|market_volatility |RG01     |price volatility|
|ME00003 |2025-02-25|promotion         |RG07     |price volatility|
|ME00004 |2025-01-10|market_volatility |RG03     |local promotion |
|ME00005 |2025-01-10|festival          |RG03     |NULL            |
|ME00006 |2025-05-12|promotion         |RG06     |local promotion |
|ME00007 |2025-01-15|import_duty_change|RG05     |high demand     |
|ME00008 |2025-04-17|festival          |RG02     |local promotion |
|ME00009 |2025-02-01|import_duty_change|RG01

In [61]:
# checking null values & empty strings
from pyspark.sql import functions as F

for column in df_events.columns:
    null_count = df_events.filter(
        F.col(column).isNull()
    ).count()

    empty_count = df_events.filter(
        F.trim(F.col(column)) == ""
    ).count()

    print(
        f"{column}: NULLs = {null_count}, "
        f"Empty strings = {empty_count}"
    )

event_id: NULLs = 0, Empty strings = 0
event_date: NULLs = 0, Empty strings = 0
event_type: NULLs = 0, Empty strings = 0
region_id: NULLs = 0, Empty strings = 0
notes: NULLs = 65, Empty strings = 0


In [62]:
# The notes column contains 65 NULL values.
# Since notes are supplementary descriptive information and are not
# essential for identifying or categorizing an event, retain these
# records with NULL notes. No replacement or quarantine is required.

In [63]:
# checking duplicate event_id
df_events.groupBy("event_id") \
    .count() \
    .filter(F.col("count") > 1) \
    .show(truncate=False)

+--------+-----+
|event_id|count|
+--------+-----+
+--------+-----+



In [65]:
# validate event_dates
df_events.filter(
    F.try_to_timestamp(
        F.col("event_date"),
        F.lit("yyyy-MM-dd")
    ).isNull()
).select("event_id", "event_date").show(truncate=False)

+--------+----------+
|event_id|event_date|
+--------+----------+
|ME00018 |bad_date  |
+--------+----------+



In [66]:
# ME00018 contains an invalid event_date value ('bad_date').
# Since a valid date is essential for time-based event analysis,
# quarantine this record while preserving the original value
# and recording the reason as INVALID_EVENT_DATE.

In [67]:
# checking event_type
df_events.groupBy("event_type") \
    .count() \
    .orderBy("event_type") \
    .show(truncate=False)

+------------------+-----+
|event_type        |count|
+------------------+-----+
|festival          |76   |
|import_duty_change|71   |
|market_volatility |68   |
|promotion         |70   |
|wedding_season    |65   |
+------------------+-----+



In [68]:
# chekcing region_id
df_events.groupBy("region_id") \
    .count() \
    .orderBy("region_id") \
    .show(truncate=False)

+---------+-----+
|region_id|count|
+---------+-----+
|RG01     |39   |
|RG02     |52   |
|RG03     |39   |
|RG04     |49   |
|RG05     |41   |
|RG06     |38   |
|RG07     |49   |
|RG08     |42   |
|RG77     |1    |
+---------+-----+



In [69]:
# Region IDs are present in the market events dataset.
# Referential integrity against regional_market will be validated
# separately during cross-file validation.

In [70]:
for column in df_events.columns:
    count = df_events.filter(
        F.col(column).isNotNull() &
        (F.col(column) != F.trim(F.col(column)))
    ).count()

    print(f"{column}: {count}")

event_id: 0
event_date: 0
event_type: 0
region_id: 0
notes: 0


**CROSS FILE VALIDATIONS**

**Transaction region references**

In [73]:
# Retrieve the required DataFrames from the existing dictionary.

df_transactions = dfs["gold_transactions"]
df_regional = dfs["regional_market"]

print("Transactions DataFrame:")
df_transactions.printSchema()

print("\nRegional Market DataFrame:")
df_regional.printSchema()

Transactions DataFrame:
root
 |-- transaction_id: string (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- region_id: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- purity: string (nullable = true)
 |-- weight_grams: string (nullable = true)
 |-- quoted_rate_per_10g: string (nullable = true)
 |-- making_charge_pct: string (nullable = true)
 |-- transaction_status: string (nullable = true)


Regional Market DataFrame:
root
 |-- region_id: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- state: string (nullable = true)
 |-- making_charge_pct: string (nullable = true)



In [74]:
from pyspark.sql import functions as F

# Step 1: Select distinct region IDs from both datasets.
# We only need unique IDs for referential-integrity validation.

transaction_regions = (
    df_transactions
    .select("region_id")
    .distinct()
)

reference_regions = (
    df_regional
    .select("region_id")
    .distinct()
)


# Step 2: Find transaction region IDs that do not exist
# in the regional reference dataset.

unmatched_transaction_regions = (
    transaction_regions
    .join(
        reference_regions,
        on="region_id",
        how="left_anti"
    )
)


# Step 3: Display unmatched region IDs.

print("Unmatched region IDs in gold_transactions:")
unmatched_transaction_regions.show(truncate=False)


# Step 4: Count unmatched region IDs.

print(
    "Total unmatched region IDs:",
    unmatched_transaction_regions.count()
)

Unmatched region IDs in gold_transactions:
+---------+
|region_id|
+---------+
|RG99     |
+---------+

Total unmatched region IDs: 1


In [75]:
# RG99 is present in gold_transactions but has no matching
# record in regional_market. Retain the transactions in the
# source data, flag them as UNMATCHED_REGION_ID, and exclude
# them only from analyses requiring regional reference data.

In [76]:
# Count transaction records associated with unmatched region IDs.
# This helps us measure the actual impact on transaction analysis.

affected_transactions = (
    df_transactions
    .join(
        unmatched_transaction_regions,
        on="region_id",
        how="inner"
    )
)

print("Transactions affected by unmatched region IDs:")
print(affected_transactions.count())

affected_transactions.show(truncate=False)

Transactions affected by unmatched region IDs:
1
+---------+--------------+----------+------------+------+------------+-------------------+-----------------+------------------+
|region_id|transaction_id|sale_date |product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|
+---------+--------------+----------+------------+------+------------+-------------------+-----------------+------------------+
|RG99     |GT000013      |2025-06-24|coin        |NULL  |100         |2797.66            |0.1              |returned          |
+---------+--------------+----------+------------+------+------------+-------------------+-----------------+------------------+



**Validate market_events against regional_market**

In [77]:
# Retrieve the market events DataFrame.
df_events = dfs["market_events"]

# Get distinct region IDs from the events and regional reference datasets.
event_regions = df_events.select("region_id").distinct()

reference_regions = df_regional.select("region_id").distinct()

# Find event region IDs that do not exist in regional_market.
unmatched_event_regions = (
    event_regions
    .join(reference_regions, on="region_id", how="left_anti")
)

print("Unmatched region IDs in market_events:")
unmatched_event_regions.show(truncate=False)

print(
    "Total unmatched event region IDs:",
    unmatched_event_regions.count()
)

Unmatched region IDs in market_events:
+---------+
|region_id|
+---------+
|RG77     |
+---------+

Total unmatched event region IDs: 1


In [79]:
# RG77 is present in market_events but has no matching record
# in regional_market. Retain the original event records, flag
# them as UNMATCHED_REGION_ID, and exclude them only from
# analyses requiring regional reference data.

In [80]:
# Identify all event records associated with unmatched region IDs.

affected_events = (
    df_events
    .join(
        unmatched_event_regions,
        on="region_id",
        how="inner"
    )
)

print("Events affected by unmatched region IDs:")
print(affected_events.count())

affected_events.show(truncate=False)

Events affected by unmatched region IDs:
1
+---------+--------+----------+------------------+----------------+
|region_id|event_id|event_date|event_type        |notes           |
+---------+--------+----------+------------------+----------------+
|RG77     |ME00013 |2025-09-30|import_duty_change|price volatility|
+---------+--------+----------+------------------+----------------+



**Validate the regional reference key**

In [81]:
# Check whether region_id is unique in the regional reference dataset.
# A region should have only one reference record.

duplicate_region_ids = (
    df_regional
    .groupBy("region_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate region IDs in regional_market:")
duplicate_region_ids.show(truncate=False)

print(
    "Total duplicate region IDs:",
    duplicate_region_ids.count()
)

Duplicate region IDs in regional_market:
+---------+-----+
|region_id|count|
+---------+-----+
+---------+-----+

Total duplicate region IDs: 0


**Check whether a regional join preserves transaction row counts**

In [82]:
# Join transactions with regional reference data.
# A left join preserves all transaction records, including unmatched regions.

transactions_before = df_transactions.count()

transactions_with_region = (
    df_transactions
    .join(
        df_regional,
        on="region_id",
        how="left"
    )
)

transactions_after = transactions_with_region.count()

print("Transaction rows before join:", transactions_before)
print("Transaction rows after join:", transactions_after)
print("Difference:", transactions_after - transactions_before)

Transaction rows before join: 2602
Transaction rows after join: 2602
Difference: 0


In [83]:
# A left join between gold_transactions and regional_market
# preserved all 2,602 transaction rows. Since region_id is unique
# in the reference dataset, the join does not multiply records.
# Unmatched region IDs are retained with NULL regional attributes.

**Validate the event-to-region join**

In [84]:
# Join events with regional reference data.
# Verify that the join preserves all original event records.

events_before = df_events.count()

events_with_region = (
    df_events
    .join(
        df_regional,
        on="region_id",
        how="left"
    )
)

events_after = events_with_region.count()

print("Event rows before join:", events_before)
print("Event rows after join:", events_after)
print("Difference:", events_after - events_before)

Event rows before join: 350
Event rows after join: 350
Difference: 0


In [85]:
# A left join between market_events and regional_market
# preserved all 350 event rows. The unique region_id key
# prevents row multiplication. Unmatched region IDs are
# retained with NULL regional attributes.

**Transaction dates vs daily price dates**

In [86]:
# Compare the date coverage of transactions and daily gold prices.
# This helps identify whether transaction dates fall outside
# the available market-price date range.

print("Transaction date range:")
df_transactions.select(
    F.min("sale_date").alias("min_date"),
    F.max("sale_date").alias("max_date"),
    F.countDistinct("sale_date").alias("distinct_dates")
).show()

print("Daily price date range:")
df_prices.select(
    F.min("price_date").alias("min_date"),
    F.max("price_date").alias("max_date"),
    F.countDistinct("price_date").alias("distinct_dates")
).show()

Transaction date range:
+----------+--------+--------------+
|  min_date|max_date|distinct_dates|
+----------+--------+--------------+
|2025-01-01| unknown|           365|
+----------+--------+--------------+

Daily price date range:
+----------+----------+--------------+
|  min_date|  max_date|distinct_dates|
+----------+----------+--------------+
|2025-01-01|2025/02/24|           365|
+----------+----------+--------------+



In [87]:
from pyspark.sql import functions as F

# Parse transaction dates.
# Invalid values such as 'unknown' will become NULL.

transactions_date_check = (
    df_transactions
    .withColumn(
        "parsed_sale_date",
        F.expr("""
            try_to_timestamp(
                trim(sale_date),
                'yyyy-MM-dd'
            )
        """).cast("date")
    )
)

# Normalize the daily price date separator before parsing.
# Both yyyy-MM-dd and yyyy/MM/dd formats are supported.

prices_date_check = (
    df_prices
    .withColumn(
        "normalized_price_date",
        F.regexp_replace(trimmed_date := F.trim(F.col("price_date")), "/", "-")
    )
    .withColumn(
        "parsed_price_date",
        F.expr("""
            try_to_timestamp(
                normalized_price_date,
                'yyyy-MM-dd'
            )
        """).cast("date")
    )
)

# Display invalid date records.

print("Invalid transaction dates:")
transactions_date_check.filter(
    F.col("parsed_sale_date").isNull()
).select(
    "transaction_id", "sale_date"
).show(truncate=False)

print("Invalid daily price dates:")
prices_date_check.filter(
    F.col("parsed_price_date").isNull()
).select(
    "price_date"
).show(truncate=False)

Invalid transaction dates:
+--------------+---------+
|transaction_id|sale_date|
+--------------+---------+
|GT000045      |unknown  |
+--------------+---------+

Invalid daily price dates:
+----------+
|price_date|
+----------+
+----------+



In [88]:
# Transaction GT000045 has an invalid sale_date ('unknown')
# and is excluded from date-based analysis. Daily price dates
# are all parseable; the alternate separator in 2025/02/24
# is normalized without quarantining the record.

**Find transaction dates with no matching daily price date**

In [89]:
# Get distinct valid transaction dates and daily price dates.
transaction_dates = (
    transactions_date_check
    .filter(F.col("parsed_sale_date").isNotNull())
    .select(F.col("parsed_sale_date").alias("sale_date"))
    .distinct()
)

price_dates = (
    prices_date_check
    .filter(F.col("parsed_price_date").isNotNull())
    .select(F.col("parsed_price_date").alias("price_date"))
    .distinct()
)

# Find transaction dates that do not exist in the daily price dataset.
unmatched_transaction_dates = (
    transaction_dates
    .join(
        price_dates,
        transaction_dates.sale_date == price_dates.price_date,
        "left_anti"
    )
)

print("Transaction dates without matching daily prices:")
unmatched_transaction_dates.show(truncate=False)

print(
    "Total unmatched transaction dates:",
    unmatched_transaction_dates.count()
)

Transaction dates without matching daily prices:
+---------+
|sale_date|
+---------+
+---------+

Total unmatched transaction dates: 0


In [90]:
# All valid transaction dates have corresponding daily price
# records. No unmatched dates were found, so valid transactions
# are eligible for date-based market price comparisons.
# GT000045 remains excluded due to its invalid sale_date.

**Check uniqueness of daily prices by date and market**

In [91]:
# Check whether each date-market combination is unique.
# Use parsed dates and trimmed market names for consistent comparison.

duplicate_daily_prices = (
    prices_date_check
    .withColumn("normalized_market", F.trim(F.col("market")))
    .groupBy("parsed_price_date", "normalized_market")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate date-market combinations:")
duplicate_daily_prices.show(truncate=False)

print(
    "Total duplicate date-market combinations:",
    duplicate_daily_prices.count()
)

Duplicate date-market combinations:
+-----------------+-----------------+-----+
|parsed_price_date|normalized_market|count|
+-----------------+-----------------+-----+
|2025-01-21       |India            |2    |
|2025-02-24       |India            |2    |
|2025-04-01       |India            |2    |
+-----------------+-----------------+-----+

Total duplicate date-market combinations: 3


In [92]:
# Inspect every record belonging to the three duplicate
# date-market combinations, including all original columns.
# This helps distinguish exact duplicates from conflicting records.

duplicate_dates = ["2025-01-21", "2025-02-24", "2025-04-01"]

duplicate_price_records = (
    prices_date_check
    .withColumn("normalized_market", F.trim(F.col("market")))
    .filter(
        F.col("parsed_price_date").isin(duplicate_dates)
        & (F.col("normalized_market") == "India")
    )
    .orderBy("parsed_price_date", "price_date")
)

duplicate_price_records.show(truncate=False)

+----------+------+-----------------+-------+------------------+------------+---------------------+-----------------+-----------------+
|price_date|market|price_per_10g_inr|usd_inr|global_gold_usd_oz|source_label|normalized_price_date|parsed_price_date|normalized_market|
+----------+------+-----------------+-------+------------------+------------+---------------------+-----------------+-----------------+
|2025-01-21|India |2709.21          |87.8   |2823.8            |daily_close |2025-01-21           |2025-01-21       |India            |
|2025-01-21|India |2709.21          |87.8   |2823.8            |daily_close |2025-01-21           |2025-01-21       |India            |
|2025-02-24|India |2732.12          |82.43  |2561.69           |daily_close |2025-02-24           |2025-02-24       |India            |
|2025/02/24|India |2761.88          |83.1   |2904.72           |daily_close |2025-02-24           |2025-02-24       |India            |
|2025-04-01|India |2863.76          |86.39  |261

In [93]:
# Exact duplicate price records for 2025-01-21 and 2025-04-01
# will be reduced to one record per date-market combination.
# The two records for 2025-02-24 contain conflicting market
# values, so both will be quarantined rather than selecting
# an arbitrary price. Original records will be preserved for review.

In [94]:
# Identify transactions occurring on the date with conflicting
# daily price records.

affected_date_transactions = (
    transactions_date_check
    .filter(F.col("parsed_sale_date") == "2025-02-24")
)

print(
    "Transactions on 2025-02-24:",
    affected_date_transactions.count()
)

affected_date_transactions.show(truncate=False)

Transactions on 2025-02-24: 2
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+----------------+
|transaction_id|sale_date |region_id|product_type|purity|weight_grams|quoted_rate_per_10g|making_charge_pct|transaction_status|parsed_sale_date|
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+----------------+
|GT000583      |2025-02-24|RG06     |coin        |18K   |5           |2900.75            |0.12             |pending           |2025-02-24      |
|GT000879      |2025-02-24|RG03     |bangle      |22k   |10          |2835.82            |0.1              |completed         |2025-02-24      |
+--------------+----------+---------+------------+------+------------+-------------------+-----------------+------------------+----------------+



In [95]:
# Two transactions (GT000583 and GT000879) occurred on
# 2025-02-24, where daily price records contain conflicting
# values. Retain both transactions, but exclude this date
# from market-price comparisons until the price conflict
# is resolved. Transaction records remain usable for other
# analyses where their required fields are valid.

In [96]:
# Calculate the current daily price record count.
print("Original daily price records:", df_prices.count())

# Count exact duplicate records across all original columns.
print(
    "Exact duplicate excess records:",
    df_prices.count() - df_prices.dropDuplicates().count()
)

# Count records belonging to the conflicting date-market key.
print(
    "Conflicting price records on 2025-02-24:",
    duplicate_price_records
    .filter(F.col("parsed_price_date") == "2025-02-24")
    .count()
)

Original daily price records: 367
Exact duplicate excess records: 2
Conflicting price records on 2025-02-24: 2


In [97]:
# Safely parse numeric price fields.
# try_cast converts malformed values to NULL instead of raising an error.

prices_numeric_check = (
    prices_date_check
    .withColumn(
        "parsed_price_inr",
        F.expr("try_cast(price_per_10g_inr AS DECIMAL(12,2))")
    )
    .withColumn(
        "parsed_usd_inr",
        F.expr("try_cast(usd_inr AS DECIMAL(12,2))")
    )
    .withColumn(
        "parsed_global_gold",
        F.expr("try_cast(global_gold_usd_oz AS DECIMAL(12,2))")
    )
)

# Display records with malformed or non-positive numeric values.

print("Invalid INR gold prices:")
prices_numeric_check.filter(
    F.col("parsed_price_inr").isNull() |
    (F.col("parsed_price_inr") <= 0)
).select(
    "price_date", "market", "price_per_10g_inr"
).show(truncate=False)

print("Invalid USD/INR exchange rates:")
prices_numeric_check.filter(
    F.col("parsed_usd_inr").isNull() |
    (F.col("parsed_usd_inr") <= 0)
).select(
    "price_date", "usd_inr"
).show(truncate=False)

print("Invalid global gold prices:")
prices_numeric_check.filter(
    F.col("parsed_global_gold").isNull() |
    (F.col("parsed_global_gold") <= 0)
).select(
    "price_date", "global_gold_usd_oz"
).show(truncate=False)

Invalid INR gold prices:
+----------+------+-----------------+
|price_date|market|price_per_10g_inr|
+----------+------+-----------------+
|2025-01-31|India |unknown          |
|2025-02-14|India |-2500            |
+----------+------+-----------------+

Invalid USD/INR exchange rates:
+----------+-------+
|price_date|usd_inr|
+----------+-------+
+----------+-------+

Invalid global gold prices:
+----------+------------------+
|price_date|global_gold_usd_oz|
+----------+------------------+
+----------+------------------+



In [100]:
# Rebuild the affected transaction records using an explicit join condition.
# Select only transaction columns to avoid duplicate column names.

affected_invalid_price_transactions = (
    transactions_date_check.alias("t")
    .join(
        invalid_price_dates.alias("p"),
        F.col("t.parsed_sale_date") == F.col("p.sale_date"),
        how="inner"
    )
    .select("t.*")
)

print(
    "Transactions on dates with invalid INR prices:",
    affected_invalid_price_transactions.count()
)

affected_invalid_price_transactions.select(
    "transaction_id",
    "sale_date",
    "region_id",
    "product_type",
    "transaction_status"
).show(truncate=False)# Two daily price records contain invalid INR gold prices:
# 'unknown' on 2025-01-31 and -2500 on 2025-02-14.
# Both records will be quarantined because a valid positive
# INR price is required for price analysis. USD/INR and global
# gold prices passed numeric validation.

Transactions on dates with invalid INR prices: 19
+--------------+----------+---------+------------+------------------+
|transaction_id|sale_date |region_id|product_type|transaction_status|
+--------------+----------+---------+------------+------------------+
|GT000039      |2025-02-14|RG07     |chain       |cancelled         |
|GT000118      |2025-02-14|RG03     |chain       |completed         |
|GT000324      |2025-01-31|RG08     |chain       |returned          |
|GT000514      |2025-02-14|RG05     |chain       |returned          |
|GT000565      |2025-02-14|RG01     |necklace    |Completed         |
|GT000790      |2025-02-14|RG01     |bar         |completed         |
|GT000973      |2025-02-14|RG04     |bangle      |cancelled         |
|GT001051      |2025-02-14|RG03     |chain       |completed         |
|GT001115      |2025-02-14|RG05     |necklace    |cancelled         |
|GT001241      |2025-01-31|RG07     |coin        |completed         |
|GT001779      |2025-01-31|RG02     |bar

In [101]:
# Nineteen transactions fall on dates with invalid daily INR
# prices: five on 2025-01-31 and fourteen on 2025-02-14.
# Retain these transactions, but exclude them from daily market
# price comparisons because the corresponding INR reference
# prices are invalid.

**Validate event dates against daily price dates**

In [102]:
# Parse event dates without modifying the original dataset.
# Invalid dates such as 'bad_date' will become NULL.

events_date_check = (
    df_events
    .withColumn(
        "parsed_event_date",
        F.expr("""
            try_to_timestamp(
                trim(event_date),
                'yyyy-MM-dd'
            )
        """).cast("date")
    )
)

# Get distinct valid event dates and daily price dates.

event_dates = (
    events_date_check
    .filter(F.col("parsed_event_date").isNotNull())
    .select(
        F.col("parsed_event_date").alias("event_date")
    )
    .distinct()
)

available_price_dates = (
    prices_date_check
    .select(
        F.col("parsed_price_date").alias("price_date")
    )
    .distinct()
)

# Find event dates with no corresponding daily price date.

unmatched_event_dates = (
    event_dates
    .join(
        available_price_dates,
        event_dates.event_date == available_price_dates.price_date,
        "left_anti"
    )
)

print("Event dates without matching daily price records:")
unmatched_event_dates.show(truncate=False)

print(
    "Total unmatched event dates:",
    unmatched_event_dates.count()
)

Event dates without matching daily price records:
+----------+
|event_date|
+----------+
|2025-02-25|
+----------+

Total unmatched event dates: 1


In [103]:
# One valid event date (2025-02-25) has no corresponding daily
# price record. Retain the event, but exclude it from price-movement
# comparisons because no market price is available for that date.

In [104]:
# Check events against unusable price dates
# Identify dates where the daily INR price cannot be reliably used.
# First, identify conflicting price records after removing exact duplicates.

distinct_price_records = (
    prices_numeric_check
    .select(
        "parsed_price_date",
        F.trim(F.col("market")).alias("market"),
        "parsed_price_inr",
        "parsed_usd_inr",
        "parsed_global_gold"
    )
    .dropDuplicates()
)

conflicting_price_dates = (
    distinct_price_records
    .groupBy("parsed_price_date", "market")
    .count()
    .filter(F.col("count") > 1)
    .select(
        F.col("parsed_price_date").alias("event_date")
    )
    .distinct()
)

# Combine conflicting dates with dates containing invalid INR prices.

invalid_inr_dates_for_events = (
    invalid_price_dates
    .select("sale_date")
    .withColumnRenamed("sale_date", "event_date")
)

unusable_price_dates = (
    conflicting_price_dates
    .union(invalid_inr_dates_for_events)
    .distinct()
)

# Find valid events falling on these unusable price dates.

events_on_unusable_price_dates = (
    events_date_check.alias("e")
    .join(
        unusable_price_dates.alias("p"),
        F.col("e.parsed_event_date") == F.col("p.event_date"),
        "inner"
    )
    .select("e.*")
)

print(
    "Events on dates with unusable INR prices:",
    events_on_unusable_price_dates.count()
)

events_on_unusable_price_dates.select(
    "event_id",
    "event_date",
    "event_type",
    "region_id"
).show(truncate=False)

Events on dates with unusable INR prices: 6
+--------+----------+------------------+---------+
|event_id|event_date|event_type        |region_id|
+--------+----------+------------------+---------+
|ME00059 |2025-02-24|festival          |RG07     |
|ME00090 |2025-02-24|promotion         |RG06     |
|ME00123 |2025-01-31|wedding_season    |RG08     |
|ME00155 |2025-01-31|import_duty_change|RG03     |
|ME00299 |2025-01-31|import_duty_change|RG04     |
|ME00337 |2025-02-24|import_duty_change|RG06     |
+--------+----------+------------------+---------+



In [105]:
# Six market events fall on dates with unusable daily INR prices:
# three on 2025-01-31 (invalid INR price) and three on 2025-02-24
# (conflicting price records). Retain the events, but exclude them
# from price-movement comparisons until a reliable price is available.

In [106]:
# Inspect the unmatched and invalid event dates
# Inspect events whose dates have no corresponding daily price record.

unmatched_event_records = (
    events_date_check
    .filter(F.col("parsed_event_date") == F.lit("2025-02-25").cast("date"))
)

print("Events on 2025-02-25 (no matching daily price):")
unmatched_event_records.select(
    "event_id",
    "event_date",
    "event_type",
    "region_id"
).show(truncate=False)


# Inspect events whose dates could not be parsed.

invalid_event_records = (
    events_date_check
    .filter(F.col("parsed_event_date").isNull())
)

print("Events with invalid dates:")
invalid_event_records.select(
    "event_id",
    "event_date",
    "event_type",
    "region_id"
).show(truncate=False)

Events on 2025-02-25 (no matching daily price):
+--------+----------+----------+---------+
|event_id|event_date|event_type|region_id|
+--------+----------+----------+---------+
|ME00003 |2025-02-25|promotion |RG07     |
+--------+----------+----------+---------+

Events with invalid dates:
+--------+----------+------------------+---------+
|event_id|event_date|event_type        |region_id|
+--------+----------+------------------+---------+
|ME00018 |bad_date  |import_duty_change|RG06     |
+--------+----------+------------------+---------+



In [107]:
# Inspect events whose dates have no corresponding daily price record.

unmatched_event_records = (
    events_date_check
    .filter(F.col("parsed_event_date") == F.lit("2025-02-25").cast("date"))
)

print("Events on 2025-02-25 (no matching daily price):")
unmatched_event_records.select(
    "event_id",
    "event_date",
    "event_type",
    "region_id"
).show(truncate=False)


# Inspect events whose dates could not be parsed.

invalid_event_records = (
    events_date_check
    .filter(F.col("parsed_event_date").isNull())
)

print("Events with invalid dates:")
invalid_event_records.select(
    "event_id",
    "event_date",
    "event_type",
    "region_id"
).show(truncate=False)

Events on 2025-02-25 (no matching daily price):
+--------+----------+----------+---------+
|event_id|event_date|event_type|region_id|
+--------+----------+----------+---------+
|ME00003 |2025-02-25|promotion |RG07     |
+--------+----------+----------+---------+

Events with invalid dates:
+--------+----------+------------------+---------+
|event_id|event_date|event_type        |region_id|
+--------+----------+------------------+---------+
|ME00018 |bad_date  |import_duty_change|RG06     |
+--------+----------+------------------+---------+



In [108]:
# Event-to-price validation identified eight affected event records:
# 1 event has no matching daily price date (ME00003),
# 1 event has an invalid date (ME00018), and 6 events fall on
# dates with invalid or conflicting INR prices.
# Retain events with valid dates, but exclude them from price
# comparisons where a reliable daily INR price is unavailable.
# Quarantine ME00018 from date-dependent processing.